# 🗺️ TavilyMap & TavilyExtract 教程

> **📚 LangChain 课程系列:构建 AI Agent 与 RAG 应用**  
> [🎓 获取完整课程](https://www.udemy.com/course/langchain/?referralCode=D981B8213164A3EA91AC)


本 notebook 演示 Tavily AI 提供的两个强大工具:
- **TavilyMap**:自动发现并绘制网站结构
- **TavilyExtract**:从网页中提取干净、结构化的内容

非常适合文档爬取、研究和内容提取! 🚀

---

## 📦 安装与环境配置

首先,让我们安装所需的包并配置环境。

In [ ]:
# 安装所需包
!pip install langchain-tavily certifi

# 用于美化打印和可视化
!pip install rich pandas

In [ ]:
import asyncio
import os
import ssl
from typing import Any, Dict, List

import certifi
from langchain_tavily import TavilyExtract, TavilyMap
from rich.console import Console
from rich.panel import Panel

# 配置 SSL 上下文(解决 Windows 证书验证问题)
ssl_context = ssl.create_default_context(cafile=certifi.where())
os.environ["SSL_CERT_FILE"] = certifi.where()
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()

# 初始化 rich console 用于美化打印
console = Console()


print("✅ 所有导入成功!")

## 🔑 API Key 配置

你需要一个 Tavily API key 来使用这些工具。前往 [tavily.com](https://app.tavily.com/home?utm_campaign=eden_marco&utm_medium=socials&utm_source=linkedin) 获取。

设置环境变量 `TAVILY_API_KEY`

In [ ]:
# 直接设置(取消注释并填入你自己的 key)
# tavily_api_key = "your_tavily_api_key_here"

# ⚠️ 安全提示:原版教程在此处硬编码了一个有效的 API Key,已经泄露,应直接在控制台吊销。
# 运行本代码前,请把下方 "在此填入你自己的 Tavily API Key" 替换为你自己在 https://app.tavily.com 申请的 key。
os.environ["TAVILY_API_KEY"] = "在此填入你自己的 Tavily API Key"

## 🗺️ TavilyMap:网站结构发现

TavilyMap 通过爬取链接来自动发现并绘制网站结构。它非常适合:
- 文档站点
- 博客归档
- 知识库
- 任何结构化网站

### 关键参数:
- `max_depth`:爬取的深度(默认:3)
- `max_breadth`:每个页面跟随多少条链接(默认:10)
- `max_pages`:发现的总页面数上限(默认:100)

In [ ]:
# 用自定义配置初始化 TavilyMap
tavily_map = TavilyMap(
    max_depth=3,        # 最多爬 3 层深度
    max_breadth=15,     # 每页最多跟随 15 条链接
    max_pages=50        # 演示用,限制到 50 个页面
)

print("✅ TavilyMap 初始化成功!")

### 📊 演示:绘制文档站点的结构

让我们绘制一个流行文档站点的结构。这里以 FastAPI 文档为例。

In [ ]:
# 要绘制的示例网站
demo_url = "https://python.langchain.com/docs/introduction/"

console.print(f"🔍 正在绘制网站结构: {demo_url}", style="bold blue")
console.print("这可能需要一点时间...")

# 绘制网站结构
site_map = tavily_map.invoke(demo_url)

# 显示结果
urls = site_map.get('results', [])
console.print(f"\n✅ 成功绘制了 {len(urls)} 个 URL!", style="bold green")

# 以前 10 个 URL 为例显示
console.print("\n📋 前 50 个发现的 URL:", style="bold yellow")
for i, url in enumerate(urls[:50], 1):
    console.print(f"  {i:2d}. {url}")

if len(urls) > 10:
    console.print(f"  ... 以及其他 {len(urls) - 50} 个 URL")

## 🔍 TavilyExtract:干净的内容提取

TavilyExtract 接收 URL 并返回干净、结构化的内容,不包含广告、导航条或其他噪音。它非常适合:
- 文档处理
- 内容分析
- 研究与数据收集
- 构建知识库

### 核心特性:
- 移除 HTML 标记和导航
- 只提取正文内容
- 处理 JavaScript 渲染的内容
- 支持批量处理

In [ ]:
# 初始化 TavilyExtract
tavily_extract = TavilyExtract()

print("✅ TavilyExtract 初始化成功!")

### 📄 演示:从 URL 提取内容

让我们从之前发现的一些 URL 中提取干净内容。

In [ ]:
# 选择几个感兴趣的 URL 用于提取
sample_urls = [urls[15]]  # 取第 16 个 URL(注意:需要 urls 足够长,否则会越界报错)
console.print(f"📚 正在从 {len(sample_urls)} 个 URL 提取内容...", style="bold blue")

# 提取内容
extraction_result = await tavily_extract.ainvoke(input={"urls": sample_urls})

# 显示结果
extracted_docs = extraction_result.get('results', [])
console.print(f"\n✅ 成功提取了 {len(extracted_docs)} 个文档!", style="bold green")

# 显示每个提取文档的摘要
for i, doc in enumerate(extracted_docs, 1):
    url = doc.get('url', '未知')
    content = doc.get('raw_content', '')

    # 为每个文档创建一个面板
    panel_content = f"""URL: {url}
内容长度: {len(content):,} 字符
预览: {content}..."""

    console.print(Panel(panel_content, title=f"文档 {i}", border_style="blue"))
    print()  # 增加间距

### ⚡ 批量处理演示

对于更大的数据集,我们可以按批次处理 URL 以优化性能并应对速率限制。

In [ ]:
def chunk_urls(urls: List[str], chunk_size: int = 3) -> List[List[str]]:
    """把 URL 列表按指定大小切分成批次。"""
    chunks = []
    for i in range(0, len(urls), chunk_size):
        chunk = urls[i:i + chunk_size]
        chunks.append(chunk)
    return chunks

async def extract_batch(urls: List[str], batch_num: int) -> List[Dict[str, Any]]:
    """从一批 URL 中提取文档。"""
    try:
        console.print(f"🔄 正在处理批次 {batch_num},包含 {len(urls)} 个 URL", style="blue")
        docs = await tavily_extract.ainvoke(input={"urls": urls})
        results = docs.get('results', [])
        console.print(f"✅ 批次 {batch_num} 完成 - 提取了 {len(results)} 个文档", style="green")
        return results
    except Exception as e:
        console.print(f"❌ 批次 {batch_num} 失败: {e}", style="red")
        return []

# 处理更大的一组 URL(分批)
url_batches = chunk_urls(urls[:9], chunk_size=3) # 取前 9 个 URL 做批处理演示,每批 3 个

console.print(f"📦 正在分批处理 9 个 URL,共 {len(url_batches)} 批", style="bold yellow")

# 并发处理所有批次
tasks = [extract_batch(batch, i + 1) for i, batch in enumerate(url_batches)]
batch_results = await asyncio.gather(*tasks)

# 展平结果
all_extracted = []
for batch_result in batch_results:
    all_extracted.extend(batch_result)

console.print(f"\n🎉 批量处理完成!总共提取的文档数: {len(all_extracted)}", style="bold green")

## 🎯 真实应用场景

以下是 TavilyMap 和 TavilyExtract 的一些实际应用:

### 1. 文档爬取
- 绘制整个文档站点
- 为搜索索引提取干净内容
- 从现有文档构建知识库

### 2. 竞品分析
- 绘制竞品网站
- 提取产品信息
- 监控内容变化

### 3. 研究与内容收集
- 从多个来源收集信息
- 构建分析用数据集
- 创建内容归档

### 4. SEO 与站点分析
- 发现站点上的所有页面
- 分析内容结构
- 识别内容缺口

## 🎯 总结

本教程演示了 TavilyMap 和 TavilyExtract 在自动化网络内容发现与提取方面的强大能力:

### 关键要点:

1. **TavilyMap** 非常适合:
   - 发现网站结构
   - 找到站点上的所有页面
   - 站点审计

2. **TavilyExtract** 擅长:
   - 干净的内容提取
   - 移除 HTML 噪音
   - 批量处理
   - 结构化数据收集

3. **两者结合** 能够:
   - 完整的文档爬取
   - 自动化内容管道
   - 创建知识库
   - 研究自动化

### 下一步:
- 与向量数据库集成以实现语义搜索
- 添加内容过滤与分类
- 构建内容变化监控系统
- 创建自动化报告仪表盘

---

**祝爬取愉快!** 🚀